In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import warnings
from sklearn.exceptions import ConvergenceWarning

# Optional: ignore convergence warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Load the preprocessed dataset
df = pd.read_csv('../results/outputs/final_preprocessed_data.csv')
X = df[['PC1', 'PC2', 'PC3', 'PC4', 'PC5']]
y = df['Total_Score']  # Target
np.random.seed(42)
noise = np.random.normal(0, y.std() * 0.5, len(y))  # 50% of target std as noise
y_noisy = y + noise

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_noisy, test_size=0.2, random_state=42)

# Create a pipeline with scaling and Lasso regression
lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', Lasso(max_iter=5000, random_state=42))
])

# Define parameter grid for alpha
param_grid_lasso = {'lasso__alpha': [0.1, 1.0, 10.0]}
grid_lasso = GridSearchCV(lasso_pipeline, param_grid_lasso, cv=3, scoring='neg_mean_squared_error')
grid_lasso.fit(X_train, y_train)

# Get the best Lasso model
best_lasso = grid_lasso.best_estimator_
y_pred_lasso = best_lasso.predict(X_test)

# Calculate performance metrics
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
r2_lasso = r2_score(y_test, y_pred_lasso)
cv_rmse_lasso = np.sqrt(-grid_lasso.best_score_)
cv_r2_lasso = cross_val_score(best_lasso, X, y_noisy, cv=3, scoring='r2').mean()

# Save results to CSV
results_lasso = pd.DataFrame({
    'RMSE (Test)': [rmse_lasso],
    'R² (Test)': [r2_lasso],
    'CV RMSE': [cv_rmse_lasso],
    'CV R²': [cv_r2_lasso],
    'Best Params': [grid_lasso.best_params_]
})
results_lasso.to_csv('../results/outputs/lasso_results_IT24102308.csv', index=False)
print("Saved results to '../results/outputs/lasso_results_IT24102308.csv'")

# Visualization: Predicted vs Actual
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_lasso, color='orange', alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Total Score')
plt.ylabel('Predicted Total Score')
plt.title('Lasso (IT24102308): Predicted vs Actual Total Score')
plt.tight_layout()
plt.savefig('../results/eda_visualizations/lasso_IT24102308_pred_vs_actual.png')
plt.close()
print("Saved plot to '../results/eda_visualizations/lasso_IT24102308_pred_vs_actual.png'")

Saved results to '../results/outputs/lasso_results_IT24102308.csv'
Saved plot to '../results/eda_visualizations/lasso_IT24102308_pred_vs_actual.png'
